# A股量化研究 — 因子分析与回测可视化

**运行前提：** 先执行 `scripts/run_pipeline.py` 生成数据

---

In [2]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from config import DATA_DIR, RESULT_DIR
from model.trainer import compute_ic_summary, compute_quintile_returns
from backtest.engine import compute_performance_metrics, compute_monthly_returns

# ── 绘图风格 ──────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 130,
    'font.family': 'SimHei',          # 中文字体
    'axes.unicode_minus': False,
    'axes.facecolor': '#f8f9fa',
    'figure.facecolor': 'white',
    'axes.grid': True,
    'grid.alpha': 0.3,
})
COLOR_PORT  = '#2563eb'
COLOR_BENCH = '#dc2626'
COLOR_POS   = '#16a34a'
COLOR_NEG   = '#dc2626'

print('✅ 环境加载完成')

OSError: libgomp.so.1: cannot open shared object file: No such file or directory

In [ ]:
# ═══════════════════════════════════════════
# 加载数据
# ═══════════════════════════════════════════
PROCESSED = DATA_DIR / 'processed'

portfolio_returns = pd.read_csv(RESULT_DIR / 'portfolio_returns.csv',
                                 index_col=0, parse_dates=True).squeeze()
monthly_table     = pd.read_csv(RESULT_DIR / 'monthly_returns.csv', index_col=0)
metrics           = pd.read_csv(RESULT_DIR / 'metrics.csv', index_col=0).iloc[0]
ic_series         = pd.read_csv(PROCESSED / 'ic_series.csv',
                                  index_col=0, parse_dates=True).squeeze()
feature_imp       = pd.read_csv(PROCESSED / 'feature_importance.csv', index_col=0)

# 基准
bench_path = DATA_DIR / 'raw' / 'benchmark_daily.parquet'
bench_ret  = None
if bench_path.exists():
    bench_df  = pd.read_parquet(bench_path)
    bench_ret = bench_df['close'].pct_change().dropna()
    bench_ret = bench_ret.reindex(portfolio_returns.index)

print(f'组合收益率数据点: {len(portfolio_returns)}')
print(f'IC序列数据点: {len(ic_series)}')
print(f'特征重要性: {len(feature_imp)} 个因子')

## 1. 绩效总览

In [ ]:
# ─── 绩效指标表格 ─────────────────────────────────────────────────
print('\n' + '='*50)
print('        A股量化策略回测报告')
print('='*50)
for k, v in metrics.items():
    print(f'  {str(k):<16}: {v}')
print('='*50)

## 2. 累计收益曲线 vs 基准

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12),
                          gridspec_kw={'height_ratios': [3, 1, 1]})

# ── 累计收益 ────────────────────────────────────────────────────────
ax1 = axes[0]
cum_port = (1 + portfolio_returns).cumprod()
ax1.plot(cum_port.index, cum_port.values, color=COLOR_PORT,
          lw=1.8, label='量化策略', zorder=3)

if bench_ret is not None:
    cum_bench = (1 + bench_ret.fillna(0)).cumprod()
    ax1.plot(cum_bench.index, cum_bench.values, color=COLOR_BENCH,
              lw=1.5, linestyle='--', label='沪深300', zorder=2)

# in/out-of-sample 分界线
from config import TRAIN_END, TEST_START
ax1.axvline(pd.Timestamp(TEST_START), color='gray',
             linestyle=':', lw=1.5, label='样本外起点')
ax1.fill_betweenx([0, ax1.get_ylim()[1] if ax1.get_ylim()[1] > 1 else 3],
                   pd.Timestamp(TEST_START), cum_port.index[-1],
                   alpha=0.05, color='green', label='样本外区间')
ax1.set_ylabel('累计净值')
ax1.set_title('累计收益曲线', fontsize=14, fontweight='bold')
ax1.legend(loc='upper left')
ax1.set_yscale('log')

# ── 回撤 ─────────────────────────────────────────────────────────────
ax2 = axes[1]
roll_max = cum_port.cummax()
drawdown = cum_port / roll_max - 1
ax2.fill_between(drawdown.index, drawdown.values, 0,
                  color=COLOR_NEG, alpha=0.5)
ax2.set_ylabel('回撤')
ax2.set_title('回撤走势', fontsize=11)
ax2.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))

# ── 月度超额收益 ────────────────────────────────────────────────────
ax3 = axes[2]
if bench_ret is not None:
    monthly_port  = (1 + portfolio_returns).resample('ME').prod() - 1
    monthly_bench = (1 + bench_ret.fillna(0)).resample('ME').prod() - 1
    excess        = monthly_port - monthly_bench.reindex(monthly_port.index).fillna(0)
    colors        = [COLOR_POS if x >= 0 else COLOR_NEG for x in excess]
    ax3.bar(excess.index, excess.values, color=colors, width=20, alpha=0.8)
    ax3.axhline(0, color='black', lw=0.8)
    ax3.set_ylabel('月度超额')
    ax3.set_title('月度超额收益', fontsize=11)
    ax3.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))

plt.tight_layout()
plt.savefig(RESULT_DIR / 'cumulative_returns.png', bbox_inches='tight', dpi=150)
plt.show()

## 3. 月度收益热力图

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

# 只显示月度收益（不含Annual列）
month_cols = [c for c in monthly_table.columns if c != 'Annual']
heat_data  = monthly_table[month_cols].astype(float)

mask = heat_data.isna()
sns.heatmap(
    heat_data * 100,
    annot=True, fmt='.1f', cmap='RdYlGn',
    center=0, vmin=-15, vmax=15,
    linewidths=0.5, ax=ax,
    mask=mask,
    cbar_kws={'label': '月收益率 (%)'}
)

# 在右侧添加年度收益
if 'Annual' in monthly_table.columns:
    annual = monthly_table['Annual'].astype(float) * 100
    for i, (yr, val) in enumerate(annual.items()):
        if not np.isnan(val):
            color = COLOR_POS if val >= 0 else COLOR_NEG
            ax.text(len(month_cols) + 0.3, i + 0.5,
                    f'{val:.1f}%', va='center', fontsize=9,
                    color=color, fontweight='bold')

ax.set_title('月度收益率热力图 (%)', fontsize=14, fontweight='bold')
ax.set_xlabel('月份')
ax.set_ylabel('年份')
plt.tight_layout()
plt.savefig(RESULT_DIR / 'monthly_heatmap.png', bbox_inches='tight', dpi=150)
plt.show()

## 4. 因子IC分析

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

ic_clean = ic_series.dropna()

# ── IC时序 ────────────────────────────────────────────────────────
ax = axes[0]
colors = [COLOR_POS if x >= 0 else COLOR_NEG for x in ic_clean]
ax.bar(ic_clean.index, ic_clean.values, color=colors, alpha=0.7, width=20)
ax.plot(ic_clean.index,
         ic_clean.rolling(6).mean(),
         color='navy', lw=1.8, label='6月滚动均值')
ax.axhline(0,    color='black', lw=0.8)
ax.axhline(0.02, color='gray',  lw=1, linestyle='--', alpha=0.7)
ax.axhline(-0.02,color='gray',  lw=1, linestyle='--', alpha=0.7)
ax.set_title('IC时序', fontsize=12, fontweight='bold')
ax.set_ylabel('IC值')
ax.legend()

# IC统计注释
stats = compute_ic_summary(ic_clean)
info  = f"均值: {stats.get('IC均值',0):.4f}\nICIR: {stats.get('ICIR',0):.3f}\nt统计: {stats.get('t统计量',0):.2f}"
ax.text(0.02, 0.98, info, transform=ax.transAxes,
         va='top', fontsize=9, bbox=dict(boxstyle='round', alpha=0.1))

# ── IC分布 ────────────────────────────────────────────────────────
ax = axes[1]
ax.hist(ic_clean, bins=30, color=COLOR_PORT, alpha=0.7, edgecolor='white')
ax.axvline(ic_clean.mean(), color=COLOR_NEG, lw=2,
            linestyle='--', label=f'均值={ic_clean.mean():.4f}')
ax.axvline(0, color='black', lw=1)
ax.set_title('IC分布', fontsize=12, fontweight='bold')
ax.set_xlabel('IC值')
ax.legend()

# ── 累计IC ───────────────────────────────────────────────────────
ax = axes[2]
cumulative_ic = ic_clean.cumsum()
ax.plot(cumulative_ic.index, cumulative_ic.values, color=COLOR_PORT, lw=1.8)
ax.fill_between(cumulative_ic.index, cumulative_ic.values, 0,
                  alpha=0.15, color=COLOR_PORT)
ax.set_title('累计IC', fontsize=12, fontweight='bold')
ax.set_ylabel('累计IC')

plt.suptitle('因子预测力分析（IC Analysis）', fontsize=14,
              fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(RESULT_DIR / 'ic_analysis.png', bbox_inches='tight', dpi=150)
plt.show()

print('\nIC统计摘要:')
for k, v in stats.items():
    print(f'  {k}: {v}')

## 5. 特征重要性

In [ ]:
if not feature_imp.empty and 'mean' in feature_imp.columns:
    top_n  = min(15, len(feature_imp))
    top_ft = feature_imp.nlargest(top_n, 'mean')

    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.barh(top_ft.index, top_ft['mean'],
                    color=COLOR_PORT, alpha=0.85)
    if 'std' in top_ft.columns:
        ax.errorbar(top_ft['mean'], top_ft.index,
                     xerr=top_ft['std'], fmt='none',
                     color='gray', capsize=3, lw=1)

    # 标注数值
    for bar, val in zip(bars, top_ft['mean']):
        ax.text(val + top_ft['mean'].max() * 0.01,
                 bar.get_y() + bar.get_height() / 2,
                 f'{val:.0f}', va='center', fontsize=9)

    ax.set_title(f'LightGBM 特征重要性 (Top {top_n})',
                  fontsize=14, fontweight='bold')
    ax.set_xlabel('重要性得分（分裂次数）')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig(RESULT_DIR / 'feature_importance.png', bbox_inches='tight', dpi=150)
    plt.show()
else:
    print('特征重要性数据不可用')

## 6. 分组收益分析（Quintile Analysis）

In [ ]:
pred_path = DATA_DIR / 'processed' / 'predictions.parquet'
panel_path = DATA_DIR / 'processed' / 'panel_with_universe.parquet'

if pred_path.exists() and panel_path.exists():
    predictions = pd.read_parquet(pred_path)
    panel       = pd.read_parquet(panel_path)
    
    from model.trainer import build_target
    target  = build_target(panel, forward_periods=21)
    q_rets  = compute_quintile_returns(predictions, target)
    
    if not q_rets.empty:
        mean_q_ret = q_rets.mean() * 100  # 转为百分比
        
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        # 分组平均收益
        ax = axes[0]
        colors = [COLOR_NEG if i < len(mean_q_ret)//2 else COLOR_POS
                   for i in range(len(mean_q_ret))]
        ax.bar(mean_q_ret.index, mean_q_ret.values,
                color=colors, alpha=0.85, width=0.6)
        ax.axhline(0, color='black', lw=1)
        ax.set_title('分组平均月收益率（Q1=最低分，Q5=最高分）',
                      fontsize=12, fontweight='bold')
        ax.set_ylabel('平均月收益率 (%)')
        for i, (label, val) in enumerate(mean_q_ret.items()):
            ax.text(i, val + (0.05 if val >= 0 else -0.15),
                     f'{val:.2f}%', ha='center', fontsize=9)
        
        # 分组累计收益
        ax = axes[1]
        cmap = plt.cm.RdYlGn
        for i, col in enumerate(q_rets.columns):
            cum = (1 + q_rets[col].dropna()).cumprod()
            color = cmap(i / max(len(q_rets.columns) - 1, 1))
            ax.plot(cum.index, cum.values, lw=1.5,
                     color=color, label=col)
        ax.set_title('分组累计收益曲线', fontsize=12, fontweight='bold')
        ax.set_ylabel('累计净值')
        ax.legend()
        ax.set_yscale('log')
        
        plt.suptitle('Quintile Analysis — 因子预测单调性验证',
                      fontsize=14, fontweight='bold', y=1.02)
        plt.tight_layout()
        plt.savefig(RESULT_DIR / 'quintile_analysis.png',
                     bbox_inches='tight', dpi=150)
        plt.show()
else:
    print('需要先运行完整pipeline')

## 7. 滚动夏普比率（策略稳定性）

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# 滚动12月夏普
ax = axes[0]
rolling_sharpe = (
    portfolio_returns.rolling(252)
    .apply(lambda r: r.mean() / r.std() * np.sqrt(252) if r.std() > 0 else 0)
)
ax.plot(rolling_sharpe.index, rolling_sharpe.values,
         color=COLOR_PORT, lw=1.5)
ax.axhline(0, color='black', lw=0.8)
ax.axhline(1, color=COLOR_POS, lw=1, linestyle='--', alpha=0.7, label='夏普=1')
ax.fill_between(rolling_sharpe.index,
                  rolling_sharpe.values, 0,
                  where=rolling_sharpe.values >= 0,
                  alpha=0.2, color=COLOR_POS)
ax.fill_between(rolling_sharpe.index,
                  rolling_sharpe.values, 0,
                  where=rolling_sharpe.values < 0,
                  alpha=0.2, color=COLOR_NEG)
ax.set_title('滚动12月夏普比率', fontsize=12, fontweight='bold')
ax.set_ylabel('夏普比率')
ax.legend()

# 年度收益柱状图
ax = axes[1]
annual_ret = (1 + portfolio_returns).resample('YE').prod() - 1
colors = [COLOR_POS if x >= 0 else COLOR_NEG for x in annual_ret]
ax.bar(annual_ret.index.year, annual_ret.values * 100,
        color=colors, alpha=0.85, width=0.6)
ax.axhline(0, color='black', lw=1)
for yr, val in zip(annual_ret.index.year, annual_ret.values * 100):
    ax.text(yr, val + (1 if val >= 0 else -3),
             f'{val:.1f}%', ha='center', fontsize=9)
ax.set_title('年度收益率', fontsize=12, fontweight='bold')
ax.set_ylabel('年化收益率 (%)')
ax.yaxis.set_major_formatter(mticker.PercentFormatter())

plt.tight_layout()
plt.savefig(RESULT_DIR / 'rolling_metrics.png', bbox_inches='tight', dpi=150)
plt.show()

## 8. 总结与结论

---

填写你自己的研究结论：

- **模型有效性**：IC均值 / ICIR 是否显著？
- **样本外表现**：样本外夏普与样本内相比如何？
- **因子贡献**：哪些因子最重要？A股特色因子是否有效？
- **主要风险**：最大回撤发生在什么市场环境？
- **改进方向**：交易成本更精细化？加入更多财务因子？
